<a href="https://colab.research.google.com/github/FrancoPanico/glioblastoma-mri-seg-mgmt/blob/Z-score/z_score_normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#importing libraries for data normalization

import nibabel as nib
import numpy as np
import pandas as pd
import shutil
import os
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import shutil
import nibabel as nib
import numpy as np
import pandas as pd

data_root = Path("/content/drive/MyDrive/Assignment_26_27/training_images")
out_root = Path("/content/normalized_scans")

with_seg_root = out_root / "with_seg"
without_seg_root = out_root / "without_seg"
with_seg_root.mkdir(parents=True, exist_ok=True)
without_seg_root.mkdir(parents=True, exist_ok=True)

SCAN_FILENAMES = [
    "t1.nii.gz",
    "t1ce.nii.gz",
    "t2.nii.gz",
    "flair.nii.gz",
]
SEG_FILENAME = "seg.nii.gz"


def zscore_foreground(volume: np.ndarray, eps: float = 1e-8) -> np.ndarray:
  vol = volume.astype(np.float32)
  mask = vol > 0
  if not np.any(mask):
    return np.zeros_like(vol, dtype=np.float32)

  fg = vol[mask]
  mean = fg.mean()
  std = fg.std()

  out = np.zeros_like(vol, dtype=np.float32)
  out[mask] = (fg - mean) / max(std, eps)
  return out


patient_dirs = sorted([p for p in data_root.iterdir() if p.is_dir()])
print(f"Found {len(patient_dirs)} patient folders in total.")

records = []

for idx, p_dir in enumerate(patient_dirs):
  seg_in = p_dir / SEG_FILENAME
  has_seg = seg_in.exists()

  target_base = with_seg_root if has_seg else without_seg_root
  p_out = target_base / p_dir.name
  p_out.mkdir(parents=True, exist_ok=True)

  # 1. Normalize the 4 MRI scans for both groups
  missing_scans = []
  for fname in SCAN_FILENAMES:
    in_path = p_dir / fname
    if not in_path.exists():
      missing_scans.append(fname)
      print(f"[Missing scan] {p_dir.name}/{fname}")
      continue

    img = nib.load(in_path)
    norm_data = zscore_foreground(img.get_fdata(dtype=np.float32))
    norm_img = nib.Nifti1Image(norm_data, affine=img.affine, header=img.header)
    nib.save(norm_img, p_out / fname)

  # 2. Copy seg.nii.gz if present
  if has_seg:
    shutil.copy2(seg_in, p_out / SEG_FILENAME)

  records.append({
      "patient_id": p_dir.name,
      "has_seg": has_seg,
      "group": "with_seg" if has_seg else "without_seg",
      "out_dir": str(p_out),
      "missing_scans": ",".join(missing_scans),
  })

  if (idx + 1) % 50 == 0 or (idx + 1) == len(patient_dirs):
    print(f"Progress: {idx + 1}/{len(patient_dirs)} patients processed.")

manifest_df = pd.DataFrame(records)
manifest_path = out_root / "patient_groups.csv"
manifest_df.to_csv(manifest_path, index=False)

print("Processing complete!")
print(manifest_df["group"].value_counts())
print(f"Saved manifest to: {manifest_path}")

Found 1384 patient folders in total.
Progress: 50/1384 patients processed.
Progress: 100/1384 patients processed.
Progress: 150/1384 patients processed.
Progress: 200/1384 patients processed.
Progress: 250/1384 patients processed.
Progress: 300/1384 patients processed.
Progress: 350/1384 patients processed.
Progress: 400/1384 patients processed.
Progress: 450/1384 patients processed.
Progress: 500/1384 patients processed.
Progress: 550/1384 patients processed.
Progress: 600/1384 patients processed.
Progress: 650/1384 patients processed.
Progress: 700/1384 patients processed.
Progress: 750/1384 patients processed.
Progress: 800/1384 patients processed.
Progress: 850/1384 patients processed.
Progress: 900/1384 patients processed.
Progress: 950/1384 patients processed.
Progress: 1000/1384 patients processed.
Progress: 1050/1384 patients processed.
Progress: 1100/1384 patients processed.
Progress: 1150/1384 patients processed.
Progress: 1200/1384 patients processed.
Progress: 1250/1384 pat